# Task 1: Install required Python libraries
%pip install -q pandas scikit-learn streamlit joblib requests

In [4]:
import pandas

import sklearn

import streamlit

import joblib

import requests



print(pandas.__version__)

print(sklearn.__version__)

print(streamlit.__version__)

print(joblib.__version__)

print(requests.__version__)

2.2.3
1.7.2
1.40.0
1.5.3
2.33.1


# Task 2: Download the dataset

In [5]:
from pathlib import Path
import requests
import pandas as pd

# Create data/ folder
Path("data").mkdir(exist_ok=True)

# Download the dataset only if it doesn't exist
csv_path = Path("data/results.csv")
if not csv_path.exists():
    url = "https://raw.githubusercontent.com/IBM-SkillsBuild-AI-Builders-Challenge/hands-on-labs/main/02_football_lab_june/04_data/results.csv"
    response = requests.get(url)
    response.raise_for_status()  # Raise an error for bad status codes
    csv_path.write_text(response.text, encoding='utf-8')
    print(f"Downloaded {csv_path}")
else:
    print(f"{csv_path} already exists, skipping download")

# Load the CSV into a pandas DataFrame
matches = pd.read_csv(csv_path, parse_dates=['date'])

# Print the shape
print(f"\nDataset shape: {matches.shape}")

# Print min and max dates
print(f"Date range: {matches['date'].min()} to {matches['date'].max()}")

# Display first 3 rows
print("\nFirst 3 rows:")
display(matches.head(3))

data\results.csv already exists, skipping download

Dataset shape: (49329, 9)
Date range: 1872-11-30 00:00:00 to 2026-06-27 00:00:00

First 3 rows:


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False


# Task 3: Explore the data

In [6]:
import pandas as pd

# 1. Top 10 most frequent tournaments
print("Top 10 most frequent tournaments:")
print(matches['tournament'].value_counts().head(10))
print()

# 2. Top 15 teams by total matches played
print("Top 15 teams by total matches played:")
home_counts = matches['home_team'].value_counts()
away_counts = matches['away_team'].value_counts()
total_matches = home_counts.add(away_counts, fill_value=0).astype(int)
print(total_matches.sort_values(ascending=False).head(15))
print()

# 3. Number of matches per decade
print("Number of matches per decade:")
matches_copy = matches.copy()
matches_copy['decade'] = (matches_copy['date'].dt.year // 10 * 10).astype(str) + 's'
decade_counts = matches_copy['decade'].value_counts().sort_index()
print(decade_counts)

Top 10 most frequent tournaments:
tournament
Friendly                                18257
Soccer World Cup qualification           8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
Soccer World Cup                         1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
Name: count, dtype: int64

Top 15 teams by total matches played:
Sweden         1102
England        1091
Argentina      1069
Brazil         1060
Germany        1032
South Korea    1008
Hungary        1004
Mexico         1003
Uruguay         973
France          936
Italy           891
Poland          890
Switzerland     885
Netherlands     880
Norway          873
Name: count, dtype: int64

Number of matches per decade:
decade
1870s      13
1880s      55
1890s      59
1900s     137
1910s     

# Task 4: Engineer features for the model

In [8]:
import pandas as pd

# Filter matches from 1990-01-01 onward and sort by date
filtered_matches = matches[matches["date"] >= "1990-01-01"].copy()
filtered_matches = filtered_matches.sort_values("date").reset_index(drop=True)

# Define major tournaments
major_tournaments = {
    "Soccer World Cup",
    "Soccer World Cup qualification",
    "UEFA Euro",
    "UEFA Euro qualification",
    "Copa América",
    "African Cup of Nations"
}

# Helper functions to compute features from history
def winrate(hist):
    """Calculate win rate from history list of (goals_for, goals_against, won) tuples."""
    if len(hist) == 0:
        return 0.5
    wins = sum(h[2] for h in hist)
    return wins / len(hist)

def goal_avg(hist):
    """Calculate average goals scored from history."""
    if len(hist) == 0:
        return 1.0
    total_goals = sum(h[0] for h in hist)
    return total_goals / len(hist)

def recent_form(hist):
    """Calculate win rate over last 10 matches."""
    if len(hist) < 10:
        return 0.5
    recent = hist[-10:]
    wins = sum(h[2] for h in recent)
    return wins / len(recent)

# Initialize team history dictionary
team_history = {}

# Build features list
features_list = []

for idx, row in filtered_matches.iterrows():
    home_team = row["home_team"]
    away_team = row["away_team"]
    home_score = row["home_score"]
    away_score = row["away_score"]
    
    # Get current history for both teams (empty list if team not seen before)
    home_hist = team_history.get(home_team, [])
    away_hist = team_history.get(away_team, [])
    
    # Compute features from current history (before this match)
    team_a_winrate = winrate(home_hist)
    team_b_winrate = winrate(away_hist)
    team_a_goal_avg = goal_avg(home_hist)
    team_b_goal_avg = goal_avg(away_hist)
    team_a_recent_form = recent_form(home_hist)
    team_b_recent_form = recent_form(away_hist)
    is_neutral = int(row["neutral"])
    is_major_tournament = 1 if row["tournament"] in major_tournaments else 0
    
    # Determine outcome
    if home_score > away_score:
        outcome = 0
    elif home_score == away_score:
        outcome = 1
    else:
        outcome = 2
    
    # Add to features list
    features_list.append({
        "date": row["date"],
        "home_team": home_team,
        "away_team": away_team,
        "team_a_winrate": team_a_winrate,
        "team_b_winrate": team_b_winrate,
        "team_a_goal_avg": team_a_goal_avg,
        "team_b_goal_avg": team_b_goal_avg,
        "team_a_recent_form": team_a_recent_form,
        "team_b_recent_form": team_b_recent_form,
        "is_neutral": is_neutral,
        "is_major_tournament": is_major_tournament,
        "outcome": outcome
    })
    
    # Update history for both teams AFTER computing features
    # Home team perspective
    home_won = 1 if home_score > away_score else 0
    if home_team not in team_history:
        team_history[home_team] = []
    team_history[home_team].append((home_score, away_score, home_won))
    
    # Away team perspective
    away_won = 1 if away_score > home_score else 0
    if away_team not in team_history:
        team_history[away_team] = []
    team_history[away_team].append((away_score, home_score, away_won))

# Create DataFrame
features_df = pd.DataFrame(features_list)

# Print shape
print(features_df.shape)

# Display first 3 rows (bare expression for HTML rendering)
features_df.head(3)

(32212, 12)


,date,home_team,away_team,team_a_winrate,team_b_winrate,team_a_goal_avg,team_b_goal_avg,team_a_recent_form,team_b_recent_form,is_neutral,is_major_tournament,outcome
0,1990-01-12,Algeria,Mali,0.5,0.5,1.0,1.0,0.5,0.5,1,0,0
1,1990-01-14,Algeria,Cameroon,1.0,0.5,5.0,1.0,0.5,0.5,1,0,0
2,1990-01-17,Greece,Belgium,0.5,0.5,1.0,1.0,0.5,0.5,0,0,0


# Task 5: Split data into training and test sets

In [9]:
import pandas as pd

# Define feature columns
feature_cols = [
    "team_a_winrate",
    "team_b_winrate",
    "team_a_goal_avg",
    "team_b_goal_avg",
    "team_a_recent_form",
    "team_b_recent_form",
    "is_neutral",
    "is_major_tournament"
]

# Time-based split: training before 2018-01-01, test on or after 2018-01-01
train_mask = features_df["date"] < "2018-01-01"
test_mask = features_df["date"] >= "2018-01-01"

# Create training and test sets
X_train = features_df.loc[train_mask, feature_cols]
X_test = features_df.loc[test_mask, feature_cols]
y_train = features_df.loc[train_mask, "outcome"]
y_test = features_df.loc[test_mask, "outcome"]

# Print shapes
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print()

# Print class distribution in training set
print("Class distribution in y_train:")
print(y_train.value_counts(normalize=True))

X_train shape: (24179, 8)
X_test shape: (8033, 8)
y_train shape: (24179,)
y_test shape: (8033,)

Class distribution in y_train:
outcome
0    0.487117
2    0.276273
1    0.236610
Name: proportion, dtype: float64


# Task 6: Train and evaluate the model

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import pandas as pd
import numpy as np

# Create and train the model
model = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Calculate and print test accuracy
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {test_accuracy * 100:.2f}%")
print()

# Calculate and print baseline accuracy (always predict most frequent class)
most_frequent_class = y_train.value_counts().idxmax()
baseline_accuracy = (y_test == most_frequent_class).mean()
print(f"Baseline accuracy (always predict class {most_frequent_class}): {baseline_accuracy * 100:.2f}%")
print()

# Print confusion matrix with labels
cm = confusion_matrix(y_test, y_pred)
labels = ["Home win", "Draw", "Away win"]
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print("Confusion Matrix (rows=actual, columns=predicted):")
print(cm_df)
print()

# Print feature importances sorted descending
print("Feature importances (sorted descending):")
importances = model.feature_importances_
feature_importance_pairs = list(zip(feature_cols, importances))
feature_importance_pairs.sort(key=lambda x: x[1], reverse=True)
for feature, importance in feature_importance_pairs:
    print(f"{feature}: {importance:.4f}")

Test accuracy: 56.02%

Baseline accuracy (always predict class 0): 47.19%

Confusion Matrix (rows=actual, columns=predicted):
          Home win  Draw  Away win
Home win      3393    32       366
Draw          1394    31       413
Away win      1273    55      1076

Feature importances (sorted descending):
team_b_winrate: 0.2211
team_a_winrate: 0.2056
team_b_goal_avg: 0.1882
team_a_goal_avg: 0.1826
team_b_recent_form: 0.0752
team_a_recent_form: 0.0751
is_neutral: 0.0274
is_major_tournament: 0.0248


# Task 7: Save the model and team statistics

In [11]:
from pathlib import Path
import joblib
import pandas as pd

# Create models/ directory
Path("models").mkdir(exist_ok=True)

# Build set of soccer teams (teams that appear in World Cup qualification)
wc_qual_matches = matches[matches["tournament"] == "Soccer World Cup qualification"]
soccer_teams = set(wc_qual_matches["home_team"]) | set(wc_qual_matches["away_team"])

# Build team_stats dictionary
team_stats = {}

# Get all unique teams
all_teams = set(matches["home_team"]) | set(matches["away_team"])

for team in all_teams:
    # Skip teams not in soccer_teams
    if team not in soccer_teams:
        continue
    
    # Get all matches for this team (home and away)
    home_matches = matches[matches["home_team"] == team].copy()
    away_matches = matches[matches["away_team"] == team].copy()
    
    # Calculate total matches
    total_matches = len(home_matches) + len(away_matches)
    
    # Skip teams with fewer than 30 matches
    if total_matches < 30:
        continue
    
    # Calculate wins
    home_wins = (home_matches["home_score"] > home_matches["away_score"]).sum()
    away_wins = (away_matches["away_score"] > away_matches["home_score"]).sum()
    total_wins = home_wins + away_wins
    
    # Calculate winrate
    winrate_val = total_wins / total_matches
    
    # Calculate total goals scored
    home_goals = home_matches["home_score"].sum()
    away_goals = away_matches["away_score"].sum()
    total_goals = home_goals + away_goals
    
    # Calculate goal average
    goal_avg_val = total_goals / total_matches
    
    # Calculate recent form (last 10 matches by date)
    # Combine home and away matches and sort by date
    home_matches["is_home"] = True
    away_matches["is_home"] = False
    all_team_matches = pd.concat([home_matches, away_matches]).sort_values("date")
    
    if len(all_team_matches) < 10:
        recent_form_val = 0.5
    else:
        last_10 = all_team_matches.tail(10)
        recent_wins = 0
        for _, match in last_10.iterrows():
            if match["is_home"]:
                if match["home_score"] > match["away_score"]:
                    recent_wins += 1
            else:
                if match["away_score"] > match["home_score"]:
                    recent_wins += 1
        recent_form_val = recent_wins / 10
    
    # Store team stats
    team_stats[team] = {
        "winrate": float(winrate_val),
        "goal_avg": float(goal_avg_val),
        "recent_form": float(recent_form_val),
        "matches_played": int(total_matches)
    }

# Save the model
joblib.dump(model, "models/match_predictor.pkl")

# Save team data and feature columns
joblib.dump({"team_stats": team_stats, "feature_cols": feature_cols}, "models/team_data.pkl")

# Print number of teams stored
print(f"Number of teams stored: {len(team_stats)}")
print()

# Print top 5 teams by winrate (among teams with >= 100 matches)
print("Top 5 teams by winrate (among teams with >= 100 matches):")
teams_100plus = {team: stats for team, stats in team_stats.items() if stats["matches_played"] >= 100}
sorted_teams = sorted(teams_100plus.items(), key=lambda x: x[1]["winrate"], reverse=True)
for i, (team, stats) in enumerate(sorted_teams[:5], 1):
    print(f"{i}. {team}: {stats['winrate']:.4f} (winrate), {stats['matches_played']} matches")

Number of teams stored: 215

Top 5 teams by winrate (among teams with >= 100 matches):
1. Brazil: 0.6321 (winrate), 1060 matches
2. Spain: 0.5867 (winrate), 784 matches
3. Germany: 0.5785 (winrate), 1032 matches
4. England: 0.5710 (winrate), 1091 matches
5. Iran: 0.5661 (winrate), 613 matches


# Task 8: Try the model

In [12]:
import pandas as pd

def predict_match(team_a, team_b, is_neutral=True, is_major_tournament=True):
    """
    Predict match outcome probabilities between two teams.
    
    Args:
        team_a: Name of team A (home team)
        team_b: Name of team B (away team)
        is_neutral: Whether the match is on neutral ground (default True)
        is_major_tournament: Whether it's a major tournament (default True)
    
    Returns:
        Dictionary with win probabilities for team_a, draw, and team_b
    """
    # Check if both teams are in team_stats
    if team_a not in team_stats:
        raise ValueError(f"Team '{team_a}' not found in team statistics. Please choose a team with at least 30 international matches in World Cup qualification.")
    if team_b not in team_stats:
        raise ValueError(f"Team '{team_b}' not found in team statistics. Please choose a team with at least 30 international matches in World Cup qualification.")
    
    # Get team statistics
    stats_a = team_stats[team_a]
    stats_b = team_stats[team_b]
    
    # Build feature row
    feature_row = pd.DataFrame([{
        "team_a_winrate": stats_a["winrate"],
        "team_b_winrate": stats_b["winrate"],
        "team_a_goal_avg": stats_a["goal_avg"],
        "team_b_goal_avg": stats_b["goal_avg"],
        "team_a_recent_form": stats_a["recent_form"],
        "team_b_recent_form": stats_b["recent_form"],
        "is_neutral": int(is_neutral),
        "is_major_tournament": int(is_major_tournament)
    }])
    
    # Reindex to match feature_cols order
    feature_row = feature_row[feature_cols]
    
    # Get prediction probabilities
    proba = model.predict_proba(feature_row)
    
    # Return probabilities (0=team_a win, 1=draw, 2=team_b win)
    return {
        "team_a_win_prob": float(proba[0][0]),
        "draw_prob": float(proba[0][1]),
        "team_b_win_prob": float(proba[0][2])
    }

# Test the function
print("Prediction for Brazil vs Argentina:")
result1 = predict_match("Brazil", "Argentina")
print(result1)
print()

print("Prediction for Germany vs Brazil:")
result2 = predict_match("Germany", "Brazil")
print(result2)

Prediction for Brazil vs Argentina:
{'team_a_win_prob': 0.39249523062963937, 'draw_prob': 0.20874109605179025, 'team_b_win_prob': 0.3987636733185705}

Prediction for Germany vs Brazil:
{'team_a_win_prob': 0.617831305753797, 'draw_prob': 0.1445198132880299, 'team_b_win_prob': 0.23764888095817305}


# Task 9: Create an interactive UI application

In [13]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
from pathlib import Path

# Configure page
st.set_page_config(
    page_title="Soccer 2026 Match Predictor",
    page_icon="⚽",
    layout="centered"
)

@st.cache_resource
def load_artifacts():
    """Load model and team data from disk."""
    model = joblib.load("models/match_predictor.pkl")
    team_data = joblib.load("models/team_data.pkl")
    team_stats = team_data["team_stats"]
    feature_cols = team_data["feature_cols"]
    return model, team_stats, feature_cols

# Load artifacts
model, team_stats, feature_cols = load_artifacts()

# Title and caption
st.title("⚽ Soccer 2026 Match Predictor")
st.caption("Predictions based on historical international football results")

# Sort team names
team_names = sorted(team_stats.keys())

# Team selection in two columns
col1, col2 = st.columns(2)

with col1:
    default_a = team_names.index("Brazil") if "Brazil" in team_names else 0
    team_a = st.selectbox("Team A", team_names, index=default_a)

with col2:
    default_b = team_names.index("Argentina") if "Argentina" in team_names else 1
    team_b = st.selectbox("Team B", team_names, index=default_b)

# Match settings
is_neutral = st.checkbox("Neutral venue", value=True)
is_major_tournament = st.checkbox("Major tournament (e.g. World Cup)", value=True)

# Predict button
if st.button("Predict", type="primary", use_container_width=True):
    if team_a == team_b:
        st.error("Please pick two different teams.")
    else:
        # Get team statistics
        stats_a = team_stats[team_a]
        stats_b = team_stats[team_b]
        
        # Build feature row
        feature_row = pd.DataFrame([{
            "team_a_winrate": stats_a["winrate"],
            "team_b_winrate": stats_b["winrate"],
            "team_a_goal_avg": stats_a["goal_avg"],
            "team_b_goal_avg": stats_b["goal_avg"],
            "team_a_recent_form": stats_a["recent_form"],
            "team_b_recent_form": stats_b["recent_form"],
            "is_neutral": int(is_neutral),
            "is_major_tournament": int(is_major_tournament)
        }])
        
        # Reindex to match feature_cols order
        feature_row = feature_row[feature_cols]
        
        # Get prediction probabilities
        proba = model.predict_proba(feature_row)
        team_a_prob = proba[0][0]
        draw_prob = proba[0][1]
        team_b_prob = proba[0][2]
        
        # Display metrics
        st.subheader("Prediction")
        metric_col1, metric_col2, metric_col3 = st.columns(3)
        
        with metric_col1:
            st.metric(f"{team_a} wins", f"{team_a_prob * 100:.1f}%")
        
        with metric_col2:
            st.metric("Draw", f"{draw_prob * 100:.1f}%")
        
        with metric_col3:
            st.metric(f"{team_b} wins", f"{team_b_prob * 100:.1f}%")
        
        # Display progress bars
        st.progress(team_a_prob, text=f"{team_a} win probability")
        st.progress(draw_prob, text="Draw probability")
        st.progress(team_b_prob, text=f"{team_b} win probability")
        
        # Display team statistics table
        st.subheader("Team Statistics")
        stats_table = pd.DataFrame({
            "Team": [team_a, team_b],
            "Win Rate": [f"{stats_a['winrate']:.3f}", f"{stats_b['winrate']:.3f}"],
            "Avg Goals Scored": [f"{stats_a['goal_avg']:.2f}", f"{stats_b['goal_avg']:.2f}"],
            "Recent Form (Last 10)": [f"{stats_a['recent_form']:.3f}", f"{stats_b['recent_form']:.3f}"],
            "Matches Played": [stats_a["matches_played"], stats_b["matches_played"]]
        })
        st.table(stats_table)

Writing app.py


# Task 10: Launch the UI application

In [14]:
import subprocess
import sys
import time
import webbrowser

# Start Streamlit as a background process
streamlit_process = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py", 
     "--server.headless", "true", 
     "--server.port", "8501", 
     "--browser.gatherUsageStats", "false"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

# Wait for server to start
print("Starting Streamlit server...")
time.sleep(4)

# Open in browser
webbrowser.open("http://localhost:8501")

# Print confirmation
print("Streamlit app is running at http://localhost:8501")
print("To stop the server later, run: streamlit_process.terminate()")

Starting Streamlit server...
Streamlit app is running at http://localhost:8501
To stop the server later, run: streamlit_process.terminate()
